# 04 — Model Evaluation

Loads the models trained in `03_train.ipynb` and evaluates **all eight** of them on the
untouched test set under a strict, leak-free protocol.

**Metrics:**
- **Rating prediction** — RMSE, MAE over the full test set.
- **Ranking** — Precision@K, Recall@K, F1@K under the *sampled-negatives* protocol:
  each user's relevant test items (rating ≥ 4.0) are ranked against 100 random
  non-relevant items, macro-averaged over a 1,000-user sample, K ∈ {5, 10, 20}.
  Candidate order is shuffled so tied predictions break randomly, and F1 is the
  harmonic mean of the macro-averaged precision and recall (so F1 always lies
  between the two).

**Models:** Global Mean · Popularity · Content-Based · User-kNN · Item-kNN ·
SVD · Weighted Hybrid · Stacked Hybrid.


In [ ]:
import sys
sys.path.insert(0, "../src")

import warnings
warnings.filterwarnings("ignore")

import json
import time
import numpy as np
import pandas as pd
import plotly.express as px
from pathlib import Path

from hybrid_recsys.pipeline.data import load_processed
from hybrid_recsys.pipeline.splits import load_splits
from hybrid_recsys.models.content import ContentBasedRecommender
from hybrid_recsys.models.collaborative import SVDModel, ItemKNNModel, UserKNNModel
from hybrid_recsys.models.hybrid import WeightedHybrid, StackedHybrid
from hybrid_recsys.evaluation.metrics import (
    evaluate_rating_prediction,
    evaluate_ranking_sampled,
)
from hybrid_recsys.config import ARTIFACTS_METRICS

FIGURES_DIR = Path("../artifacts/figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def save_fig(fig, name: str) -> None:
    fig.write_html(str(FIGURES_DIR / f"{name}.html"))
    try:
        fig.write_image(str(FIGURES_DIR / f"{name}.png"), width=1200, height=600, scale=2)
    except Exception:
        pass  # kaleido not installed — HTML only
    fig.show()

ARTIFACTS_METRICS.mkdir(parents=True, exist_ok=True)

EVAL_USERS   = 1_000   # users sampled for ranking evaluation
N_NEGATIVES  = 100     # sampled non-relevant items per user for ranking metrics
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)


## 1. Load Data & Trained Models

In [ ]:
movies           = load_processed("movies")
train, val, test = load_splits()
train_val        = pd.concat([train, val], ignore_index=True)

print(f"Test: {len(test):>10,} ratings | {test['userId'].nunique():>7,} users")

# User histories from train (content model & weighted hybrid need these).
user_ratings_map: dict = (
    train.groupby("userId")
    .apply(lambda df: dict(zip(df["movieId"], df["rating"])))
    .to_dict()
)

# Load every model trained in notebook 03.
cb       = ContentBasedRecommender.load()
user_knn = UserKNNModel.load()
item_knn = ItemKNNModel.load()
svd      = SVDModel.load()
weighted = WeightedHybrid.load()
stacked  = StackedHybrid.load()
print("Loaded 6 trained models.")

# Stratified user sample for ranking evaluation.
eval_user_ids = rng.choice(
    test["userId"].unique(),
    size=min(EVAL_USERS, test["userId"].nunique()),
    replace=False,
)
test_sample = test[test["userId"].isin(eval_user_ids)]
print(f"Ranking evaluation user sample: {len(eval_user_ids):,}")


## 2. Evaluation Helper

A shared wrapper computes both rating-prediction and ranking metrics for any
`predict_fn(user_id, movie_id) -> float`, checkpointing `all_metrics` after every model.


In [ ]:
all_metrics: dict = {}
metrics_path = ARTIFACTS_METRICS / "all_metrics.json"


def checkpoint_metrics() -> None:
    """Persist all_metrics after every model so partial results survive crashes."""
    metrics_path.write_text(json.dumps(all_metrics, indent=2))


def eval_model(key: str, label: str, predict_fn) -> dict:
    print(f"Evaluating: {label} ...")
    t0 = time.perf_counter()

    preds = np.array([predict_fn(r.userId, r.movieId) for r in test.itertuples()])
    rp    = evaluate_rating_prediction(test["rating"].values, preds)
    t_rating = time.perf_counter() - t0

    ranking = evaluate_ranking_sampled(
        test_sample, predict_fn, train_val,
        all_movie_ids=movies["movieId"].values,
        n_negatives=N_NEGATIVES,
        k_values=[5, 10, 20],
        random_state=RANDOM_STATE,
    )
    t_total = time.perf_counter() - t0

    metrics = {
        "rmse": round(rp["rmse"], 4),
        "mae":  round(rp["mae"],  4),
        **{
            f"k{k}": {m: round(v, 4) for m, v in kv.items()}
            for k, kv in ranking.items()
        },
    }
    all_metrics[label] = metrics
    checkpoint_metrics()
    print(
        f"  RMSE={metrics['rmse']}  MAE={metrics['mae']}  F1@10={metrics['k10']['f1']}"
        f"  (rating {t_rating:.1f}s · total {t_total:.1f}s)"
    )
    return metrics


## 3. Naive Baselines — Global Mean & Popularity

Recomputed from the training set (trivial, so not persisted as artifacts). The
**global mean** assigns the training average to every pair; **popularity** scores
items by interaction count. These define the floor every model must beat.


In [ ]:
global_mean     = float(train["rating"].mean())
item_popularity = train.groupby("movieId").size().to_dict()
max_pop         = max(item_popularity.values())

# Popularity is fundamentally a ranking signal; we map raw counts to the
# [0.5, 5.0] rating range so RMSE/MAE are computable. The map is monotonic,
# so ranking metrics are unaffected.
def pop_score(_user, movie_id):
    return 0.5 + 4.5 * (item_popularity.get(movie_id, 0) / max_pop)

eval_model("global_mean", "Global Mean", lambda u, i: global_mean)
eval_model("popularity",  "Popularity",  pop_score)


## 4. Content-Based

In [ ]:
eval_model("content", "Content-Based", lambda u, i: cb.predict(user_ratings_map.get(u, {}), i))


## 5. User-Based k-NN

In [ ]:
eval_model("user_knn", "User-Based k-NN", lambda u, i: user_knn.predict(u, i))


## 6. Item-Based k-NN

In [ ]:
eval_model("item_knn", "Item-Based k-NN", lambda u, i: item_knn.predict(u, i))


## 7. SVD

In [ ]:
eval_model("svd", "SVD", lambda u, i: svd.predict(u, i))


## 8. Weighted Hybrid

In [ ]:
eval_model(
    "weighted_hybrid", "Weighted Hybrid",
    lambda u, i: weighted.predict(u, i, user_ratings_map.get(u, {})),
)


## 9. Stacked Hybrid

Builds the four base predictions on the fly and feeds them through the saved Ridge
meta-model via `predict_one` — exactly the path the Streamlit serving bundle uses.
Returns the global mean if any base prediction is NaN.


In [ ]:
def stacked_predict(user_id, movie_id) -> float:
    base = np.array([
        cb.predict(user_ratings_map.get(user_id, {}), movie_id),
        user_knn.predict(user_id, movie_id),
        item_knn.predict(user_id, movie_id),
        svd.predict(user_id, movie_id),
    ], dtype=float)
    return stacked.predict_one(user_id, movie_id, base)

eval_model("stacked_hybrid", "Stacked Hybrid", stacked_predict)


## 10. Results Summary

In [ ]:
rows = []
for label, m in all_metrics.items():
    rows.append({
        "Model":  label,
        "RMSE":   m["rmse"],  "MAE":   m["mae"],
        "P@5":    m["k5"]["precision"],  "R@5":  m["k5"]["recall"],  "F1@5":  m["k5"]["f1"],
        "P@10":   m["k10"]["precision"], "R@10": m["k10"]["recall"], "F1@10": m["k10"]["f1"],
        "P@20":   m["k20"]["precision"], "R@20": m["k20"]["recall"], "F1@20": m["k20"]["f1"],
    })

results = pd.DataFrame(rows).set_index("Model")
display(
    results.style
    .highlight_min(subset=["RMSE", "MAE"],              color="#d4edda")
    .highlight_max(subset=["F1@5", "F1@10", "F1@20"],   color="#d4edda")
    .format("{:.4f}")
)


## 11. Visualisations

In [ ]:
df_plot = results.reset_index()

# RMSE / MAE grouped bar
fig1 = px.bar(
    df_plot.melt(id_vars="Model", value_vars=["RMSE", "MAE"]),
    x="Model", y="value", color="variable", barmode="group",
    title="RMSE and MAE by Model",
    labels={"value": "Error", "variable": "Metric"},
)
fig1.update_layout(xaxis_tickangle=-30)
save_fig(fig1, "08_rmse_mae")

# F1@10 bar
fig2 = px.bar(
    df_plot.sort_values("F1@10", ascending=False),
    x="Model", y="F1@10",
    title="F1@10 by Model",
    color="F1@10", color_continuous_scale="Teal", text_auto=".4f",
)
fig2.update_layout(coloraxis_showscale=False, xaxis_tickangle=-30)
save_fig(fig2, "09_f1_at_10")

# Ranking metrics @ K for best model
best_label = df_plot.sort_values("F1@10", ascending=False).iloc[0]["Model"]
best_data  = [
    {"K": k, "Metric": m.capitalize(), "Value": all_metrics[best_label][f"k{k}"][m]}
    for k in [5, 10, 20]
    for m in ["precision", "recall", "f1"]
]
fig3 = px.line(
    pd.DataFrame(best_data), x="K", y="Value", color="Metric",
    markers=True,
    title=f"Ranking Metrics @ K — {best_label}",
    labels={"Value": "Score"},
)
save_fig(fig3, "10_ranking_metrics_k")


## 12. Save Metrics

In [ ]:
checkpoint_metrics()  # final flush (each eval_model already checkpoints)
print(f"Metrics → {metrics_path}")


## Conclusion

- All eight models were evaluated on the same temporal test split under a strictly leak-free protocol.
- Rating accuracy (RMSE/MAE) and ranking quality (P/R/F1@K) are persisted to `artifacts/metrics/all_metrics.json` for the Streamlit comparison tab.
- Ranking uses **random tie-breaking** and a **consistent harmonic-mean F1**, so constant-output models (e.g. Global Mean) no longer score artificially high.
